# ddldelta quickstart

`ddldelta` is an offline **DDL-diff migration-plan generator**. Give it two
states of a schema — each state a directory of `CREATE TABLE` files, in
whatever dialect a supplier happens to deliver (T-SQL, MySQL, or any other
dialect [sqlglot](https://github.com/tobymao/sqlglot) can read) — and it
turns the difference into versioned migration files for
[schemachange](https://github.com/Snowflake-Labs/schemachange) or
[Flyway](https://flywaydb.org/), or into a CI compatibility report with a
process exit code.

**It never connects to a database.** The two DDL directories are the only
source of truth; given the same two directories, `ddldelta` always produces
the same bytes.

This notebook covers the core, one-shot workflow:

1. write two schema states (old/new) to a scratch workspace,
2. parse and inspect them,
3. run a compatibility check,
4. generate schemachange migrations — including the safety "fuse" for a
   critical change,
5. see that regenerating is a no-op (idempotency).

The companion notebook, `02_workflow_snapshots_cli.ipynb`, covers the
operational workflow: snapshots, the command-line interface, and baselines.

In [1]:
import tempfile
from pathlib import Path

work = Path(tempfile.mkdtemp(prefix="ddldelta_tutorial_"))
old_dir = work / "old"
new_dir = work / "new"
print("workspace:", work.name)

workspace: ddldelta_tutorial_177us1a_


## The "old" schema state

Two tables, `customer` and `orders`, as a supplier might deliver them in
MySQL flavored DDL.

In [2]:
old_dir.mkdir()
(old_dir / "customer.sql").write_text("""\
CREATE TABLE `customer` (
  `id` int(10) unsigned NOT NULL,
  `name` varchar(100) NOT NULL,
  `email` varchar(255) DEFAULT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")
(old_dir / "orders.sql").write_text("""\
CREATE TABLE `orders` (
  `id` int(10) unsigned NOT NULL,
  `customer_id` int(10) unsigned NOT NULL,
  `amount` int(10) NOT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")
print(sorted(str(p.relative_to(work)) for p in old_dir.glob("*.sql")))

['old\\customer.sql', 'old\\orders.sql']


## The "new" schema state

The next delivery makes two changes:

- `customer` gains a nullable `phone` column — a **safe** change, nothing
  existing can break.
- `orders.amount` changes type from `int` to `decimal(10,2)` — a **critical**
  change: existing values could be truncated or misread by anything that
  still assumes an integer.

In [3]:
new_dir.mkdir()
(new_dir / "customer.sql").write_text("""\
CREATE TABLE `customer` (
  `id` int(10) unsigned NOT NULL,
  `name` varchar(100) NOT NULL,
  `email` varchar(255) DEFAULT NULL,
  `phone` varchar(30) DEFAULT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")
(new_dir / "orders.sql").write_text("""\
CREATE TABLE `orders` (
  `id` int(10) unsigned NOT NULL,
  `customer_id` int(10) unsigned NOT NULL,
  `amount` decimal(10,2) NOT NULL,
  PRIMARY KEY (`id`)
);
""", encoding="utf-8")
print(sorted(str(p.relative_to(work)) for p in new_dir.glob("*.sql")))

['new\\customer.sql', 'new\\orders.sql']


## Parse and inspect

`PathPairSource` is the source to reach for when "old" and "new" are just
two paths you already know — no delivery-directory naming convention
required. Each path may be a directory of `*.sql` files or a single `.sql`
file; `old=None` would mean "initial state" (every table is new).

Loading it parses both directories with the dialect's parser and returns a
`SchemaPair` holding the parsed `Table` objects plus a human-readable
`comparison` label.

In [4]:
from ddldelta.ddl_parser import parser_for
from ddldelta.sources import PathPairSource

parse_ddl = parser_for("mysql")
pair = PathPairSource(old_dir, new_dir, parse_ddl).load()

print(pair.comparison)
print(sorted(pair.new))
for column in pair.new["orders"].columns:
    print(column)

old -> new
['customer', 'orders']
Column(name='id', type='INT', not_null=True)
Column(name='customer_id', type='INT', not_null=True)
Column(name='amount', type='DECIMAL(10,2)', not_null=True)


## Compatibility check

`build_report` runs the same diff and policy that the renderers use, but
writes nothing — it just answers "is the new delivery backward compatible?"
with a CI-friendly exit code: `0` compatible, `1` a critical change was
found.

In [5]:
from ddldelta.policy import FusePolicy
from ddldelta.render.report import build_report

report = build_report(pair.comparison, pair.old, pair.new, FusePolicy())
print(report.text())
print("exit_code:", report.exit_code)

old -> new: 2 change(s), 1 critical
  [safe] customer: ALTER TABLE "customer" ADD COLUMN "phone" VARCHAR(30);
  [critical] orders: ALTER TABLE "orders" ALTER COLUMN "amount" SET DATA TYPE DECIMAL(10,2);
exit_code: 1


In [6]:
print(report.to_json())

{
  "comparison": "old -> new",
  "compatible": false,
  "exit_code": 1,
  "findings": [
    {
      "table": "customer",
      "severity": "safe",
      "statement": "ALTER TABLE \"customer\" ADD COLUMN \"phone\" VARCHAR(30);"
    },
    {
      "table": "orders",
      "severity": "critical",
      "statement": "ALTER TABLE \"orders\" ALTER COLUMN \"amount\" SET DATA TYPE DECIMAL(10,2);"
    }
  ]
}


## Generate schemachange migrations

`build_plan` bundles the diff with the policy's verdict per table; a table
is critical as soon as *one* of its changes is. `SchemachangeRenderer` then
writes one versioned file per table.

Here is the safety mechanism this package is built around, "the fuse": a
table with any critical change is rendered entirely as SQL *comments*
behind a leading `SELECT 1/0 AS "CRITICAL CHANGES DETECTED...";` statement,
plus a `V<label>.0000__CRITICAL_CHANGES.sql` file summarizing every affected
table. The failing statement is deliberate — it makes the migration fail
deployment on purpose, so a human has to open the file, read the
commented-out statements, and uncomment (or rewrite) them before anything
critical can run against a real database.

In [7]:
from ddldelta.plan import build_plan
from ddldelta.render.schemachange import SchemachangeRenderer

plan = build_plan("1", pair.comparison, pair.old, pair.new, FusePolicy())
migrations_dir = work / "migrations"
written = SchemachangeRenderer(target_dir=migrations_dir, generated_by="tutorial").render(plan)
print(sorted(str(p.relative_to(work)) for p in written))

['migrations\\V1.0000__CRITICAL_CHANGES.sql', 'migrations\\V1.0001__customer.sql', 'migrations\\V1.0002__orders.sql']


In [8]:
safe_file = next(p for p in written if "customer" in p.name)
critical_file = next(p for p in written if "orders" in p.name)
summary_file = next(p for p in written if "CRITICAL_CHANGES" in p.name)

print(f"--- {safe_file.name} (SAFE) ---")
print(safe_file.read_text(encoding="utf-8"))
print(f"--- {critical_file.name} (CRITICAL - fused) ---")
print(critical_file.read_text(encoding="utf-8"))
print(f"--- {summary_file.name} (CRITICAL summary) ---")
print(summary_file.read_text(encoding="utf-8"))

--- V1.0001__customer.sql (SAFE) ---
-- generated by tutorial (old -> new); do not edit manually
ALTER TABLE "customer" ADD COLUMN "phone" VARCHAR(30);

--- V1.0002__orders.sql (CRITICAL - fused) ---
-- generated by tutorial (old -> new); do not edit manually
SELECT 1/0 AS "CRITICAL CHANGES DETECTED - PLEASE REVIEW THIS FILE!";
-- ALTER TABLE "orders" ALTER COLUMN "amount" SET DATA TYPE DECIMAL(10,2);

--- V1.0000__CRITICAL_CHANGES.sql (CRITICAL summary) ---
-- generated by tutorial (old -> new); do not edit manually
SELECT 1/0 AS "CRITICAL CHANGES DETECTED - PLEASE REVIEW THE LISTED FILES!";
-- V1.0002__orders.sql



## Idempotency

Once a migration file is written and deployed, schemachange checksums it —
rewriting it would invalidate every future deploy. `ddldelta` therefore
never overwrites a file that already exists: rendering the exact same plan
again compares the *supplier facts* of the new content against what is
already on disk, finds them identical, and silently skips every file. A
real divergence would instead raise `GenerationError` and stop.

In [9]:
again = SchemachangeRenderer(target_dir=migrations_dir, generated_by="tutorial").render(plan)
print("written on rerun:", again)

written on rerun: ()


## Where to go next

- The package [`README.md`](../README.md) covers the full command-line
  interface, the snapshot workflow, and every invariant this package
  guarantees.
- [`docs/design.md`](../docs/design.md) explains the architecture and the
  three extension seams (`PairSource`, `ChangePolicy`, `Renderer`).
- [`docs/decisions.md`](../docs/decisions.md) is the decision-by-decision
  log, including what was deliberately left out and why.
- `02_workflow_snapshots_cli.ipynb` picks up from here: running the same
  four stages through the `ddldelta` command line, keeping state across
  deliveries with snapshots, and baselining a fresh environment.